# Phase 0 go/no-go on Colab (Qwen3.5-4B)

Thin launcher: all logic lives in the repository. Runtime: **GPU (L4 or A100)**.
Run all cells; the last one prints a summary to paste back. Safe to re-run from the top.

In [6]:
REPO_URL = "https://github.com/desire-del/faithful-question-summarization.git"
REF = "main"  # use a tag (e.g. v0.0) once it exists
%cd /content
!rm -rf repo && git clone -q --depth 1 --branch $REF $REPO_URL repo
%cd /content/repo
# Colab preinstalls torchao 0.10, which peft rejects (it needs >= 0.16). This project does not use it.
!pip uninstall -y -q torchao
!pip install -q -e '.[train]'

/content
/content/repo
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for faithqs (pyproject.toml) ... done


In [7]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
NVIDIA L4, 23034 MiB


In [8]:
# 1) Generation: text-only loading, thinking disabled, latency
!python scripts/go_no_go/smoke_generate.py --env colab --model qwen35_4b

Loading weights: 100% 426/426 [00:02<00:00, 201.91it/s]
[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
{
  "check": "generati

In [9]:
# 2) GRPO: a few TRL + LoRA steps, peak memory and time per step
!python scripts/go_no_go/grpo_smoke.py --env colab --model qwen35_4b --steps 4

Loading weights: 100% 426/426 [00:00<00:00, 4753.44it/s]
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.
  0% 0/4 [00:00<?, ?it/s][transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the 

In [10]:
# 3) Summary to paste back
import json, pathlib
for path in sorted(pathlib.Path('reports/go_no_go').glob('*_colab.json')):
    report = json.loads(path.read_text())
    keep = ['decision', 'error', 'model_class', 'peak_memory_gb', 'steady_step_seconds',
            'estimated_full_run_hours', 'generate_seconds_batch', 'thinking_leaked']
    summary = {k: report[k] for k in keep if k in report}
    summary['gpu'] = (report.get('provenance', {}).get('gpu') or {}).get('name')
    summary['packages'] = report.get('provenance', {}).get('packages')
    print(path.name, json.dumps(summary, indent=1))
    for sample in report.get('samples', [])[:3]:
        print('  ->', sample['output'])

generate_qwen35_4b_colab.json {
 "decision": "go",
 "model_class": "Qwen3_5ForCausalLM",
 "peak_memory_gb": 8.52,
 "generate_seconds_batch": 3.72,
 "thinking_leaked": false,
 "gpu": "NVIDIA L4",
 "packages": {
  "faithqs": "0.0.1",
  "torch": "2.11.0+cu130",
  "transformers": "5.20.2",
  "trl": "1.19.0",
  "peft": "0.21.2",
  "accelerate": "1.15.0",
  "vllm": null
 }
}
  -> As a 34-year-old woman who is 3 months pregnant and takes levothyroxine 50 mcg daily for hypothyroidism, can I continue this medication during pregnancy?

  -> Is it safe to give my 4-year-old son ibuprofen 100mg for a fever of 102F that has lasted 3 days, or should I see a doctor?

  -> Is it normal for metformin to cause stomach cramps and diarrhea in a person with type 2 diabetes, and will these symptoms go away?

grpo_qwen35_4b_colab.json {
 "decision": "go",
 "error": null,
 "model_class": "Qwen3_5ForCausalLM",
 "peak_memory_gb": 9.2,
 "steady_step_seconds": 6.19,
 "estimated_full_run_hours": 3.1,
 "gpu": "NVID